# 🧠 Machine Learning Ultimate Reference: Part 3
## Chapters 8 - 11: Classical Machine Learning & Model Evaluation

This part covers core classical machine learning workflows:
8. **Data Preprocessing & Feature Engineering**
9. **Machine Learning Algorithms (Supervised)**
10. **Unsupervised Learning Algorithms**
11. **Model Evaluation & Hyperparameter Optimization**

---

# 🛠️ Chapter 8: Data Preprocessing & Feature Engineering

Preprocessing raw data is crucial because garbage in yields garbage out.

## 8.1 Encoding, Scaling, and Pipelines
We will build a pipeline handling numerical scaling and categorical encoding to prevent leakage.



In [ ]:
# Chapter 8.1: Preprocessing & Scaling Pipeline
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Create synthetic tabular dataset
np.random.seed(42)
data = pd.DataFrame({
    'age': np.random.normal(35, 10, 100).clip(18, 70),
    'income': np.random.normal(50000, 15000, 100).clip(20000, 100000),
    'department': np.random.choice(['Sales', 'Tech', 'HR', 'Marketing'], 100),
    'churned': np.random.choice([0, 1], 100, p=[0.7, 0.3])
})

X = data.drop('churned', axis=1)
y = data['churned']

# Define numeric vs categorical columns
numeric_features = ['age', 'income']
categorical_features = ['department']

# Numeric Pipeline (Impute then Standardize)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline (Impute then One-Hot Encode)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Build complete model pipeline
clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Pipeline
clf_pipeline.fit(X_train, y_train)
acc = clf_pipeline.score(X_test, y_test)
print(f"Pipeline fitted successfully! Test Accuracy: {acc * 100:.2f}%")



## 8.2 Common Errors & Best Practices
- **Error**: Applying `fit_transform()` on validation or test sets. This introduces target data leakage!
- **Best Practice**: Use `Pipeline` to orchestrate transformations and modeling. Only call `fit()` on the train set.

---

# 🤖 Chapter 9: Machine Learning Algorithms (Supervised)

We walk through key supervised machine learning algorithms, highlighting:
- **Linear & Logistic Regression**
- **Decision Trees & Random Forests**
- **Support Vector Machines (SVM)**
- **K-Nearest Neighbors (KNN)**
- **Naive Bayes**
- **Gradient Boosting (XGBoost, LightGBM, CatBoost)**

## 9.1 Supervised Classifiers Comparison
We write code comparing multiple classifiers on synthetic classification data.



In [ ]:
# Chapter 9.1: Supervised Classifier Comparison
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Generate classification dataset
X_clf, y_clf = make_classification(n_samples=500, n_features=10, n_informative=8, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

# Dictionary of models
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(max_depth=5),
    "SVM": SVC(kernel='rbf'),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# Evaluate all
model_scores = {}
for name, model in models.items():
    model.fit(X_train_c, y_train_c)
    preds = model.predict(X_test_c)
    model_scores[name] = accuracy_score(y_test_c, preds)

# Plot accuracy
plt.figure(figsize=(10, 5))
plt.bar(model_scores.keys(), model_scores.values(), color='darkcyan', edgecolor='black')
plt.ylabel('Accuracy')
plt.title('Classifier Accuracy Comparison')
plt.ylim(0.7, 1.0)
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle=':')
plt.show()



---

# 🏢 Chapter 10: Unsupervised Learning

Unsupervised algorithms find hidden structures in data without targets.
- **K-Means**: Centroid-based partitioning.
- **DBSCAN**: Density-based spatial clustering.
- **PCA**: Dimensionality reduction by projecting onto principal components.

## 10.1 Clustering & Dimensionality Reduction (PCA & KMeans)
We generate 2D clusters, reduce them, and assign labels using KMeans.



In [ ]:
# Chapter 10.1: PCA & KMeans Clustering
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# 1. Generate Blob Clusters in high dimension
X_blobs, y_true = make_blobs(n_samples=300, n_features=5, centers=4, cluster_std=1.5, random_state=42)

# 2. PCA to project from 5D to 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_blobs)

# 3. Apply KMeans on reduced dimensions
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans.fit(X_pca)
labels = kmeans.labels_
centers = kmeans.cluster_centers_

# 4. Visualization
plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', alpha=0.7, edgecolors='k')
plt.scatter(centers[:, 0], centers[:, 1], marker='x', s=200, linewidths=3, color='red', label='Centroids')
plt.title('KMeans Clustering on 2D PCA Space')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.grid(True)
plt.show()



---

# 📉 Chapter 11: Model Evaluation & Hyperparameter Tuning

Evaluating a model is not just checking standard accuracy. When classes are imbalanced, precision, recall, F1, and AUC are critical.

## 11.1 Metrics: Confusion Matrix, ROC-AUC



In [ ]:
# Chapter 11.1: Advanced Classification Metrics
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score
import seaborn as sns

# Model predictions (let's use the Logistic Regression model from Ch 9)
lr_model = LogisticRegression()
lr_model.fit(X_train_c, y_train_c)
y_pred_lr = lr_model.predict(X_test_c)
y_probs_lr = lr_model.predict_proba(X_test_c)[:, 1]

# 1. Confusion Matrix
cm = confusion_matrix(y_test_c, y_pred_lr)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

# 2. Classification Report
print("Classification Report:
", classification_report(y_test_c, y_pred_lr))

# 3. ROC Curve
fpr, tpr, thresholds = roc_curve(y_test_c, y_probs_lr)
auc = roc_auc_score(y_test_c, y_probs_lr)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()



## 11.2 Hyperparameter Tuning: GridSearchCV
GridSearchCV exhaustively searches over a specified parameter grid.



In [ ]:
# Chapter 11.2: Grid Search CV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, None],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_c, y_train_c)
print("Best Hyperparameters found:", grid_search.best_params_)
print(f"Best cross-validation accuracy: {grid_search.best_score_ * 100:.2f}%")



## 11.3 Exercises & Mini Quiz
### Quiz
1. What does the Area Under the ROC Curve (AUC) evaluate?
   - A) The absolute accuracy of predictions.
   - B) The model's capacity to distinguish between classes across thresholds. (Correct)
   - C) The percentage of true predictions.
2. In imbalanced classification, which metric is better when false positives are highly costly?
   - A) Recall
   - B) Precision (Correct)
   - C) Accuracy

